# Advanced Image Evaluation

ROI-aware structural and perceptual comparison of generated reconstruction candidates against an archival reference, with uncertainty and sensitivity analysis.

> Optional PyTorch/PIQ perceptual metrics are disabled by default so the notebook can run with the core environment.


In [ ]:
# 0) SETTINGS ─────────────────────────────────────────────────────────────────
USE_TORCH = False   # set False if your environment lacks torch/piq
TOP_K = 8          # how many top candidates to highlight/export


In [ ]:
# 1) IMPORTS & PATHS ───────────────────────────────────────────────────────────
from pathlib import Path
import os, glob, re, json, random, itertools
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity, peak_signal_noise_ratio
from skimage.feature import canny
from scipy import stats
from statsmodels.stats.multitest import multipletests

if USE_TORCH:
    import torch, piq

# Reproducibility
np.random.seed(42); random.seed(42)
if USE_TORCH: torch.manual_seed(42)

# Your layout
DATA_ROOT = Path("project")
REF_PATH  = DATA_ROOT/"historical.jpg"        # archival reference
CAND_DIR  = DATA_ROOT/"images"                 # all finalists anywhere under here
MASK_DIR  = DATA_ROOT/"images"/"masks"         # optional ROI masks
OUT       = DATA_ROOT/"eval_out"

for p in [OUT, OUT/"figs", OUT/"tables", OUT/"top_k"]:
    p.mkdir(parents=True, exist_ok=True)

print("REF :", REF_PATH.resolve())
print("CAND:", CAND_DIR.resolve())
print("OUT :", OUT.resolve())
assert REF_PATH.exists(), "historical.jpg not found at project root."


In [ ]:
import warnings

# These must be STRING regexes (not re.compile objects)
warnings.filterwarnings(
    "ignore",
    message=r".*The parameter 'pretrained' is deprecated.*",
    category=UserWarning,
    module=r"torchvision"
)
warnings.filterwarnings(
    "ignore",
    message=r".*Arguments other than a weight enum or `None` for 'weights'.*",
    category=UserWarning,
    module=r"torchvision"
)


In [ ]:
# 2) HELPERS ───────────────────────────────────────────────────────────────────
def imread_rgb(p):
    im = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if im is None: raise FileNotFoundError(p)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def to_float01(img):
    if img.dtype == np.uint8: return img.astype(np.float32)/255.0
    img = img.astype(np.float32); 
    return img/255.0 if img.max() > 1 else img

def resize_to(a, b_uint8):
    return cv2.resize(b_uint8, (a.shape[1], a.shape[0]), interpolation=cv2.INTER_AREA)

def load_masks(base_stem, roi_names, shape, mask_dir=MASK_DIR):
    """
    Looks for masks <stem>__<roi>.png. Each is binary (white = ROI).
    If not found, uses full image for that ROI.
    """
    masks = {}
    for roi in roi_names:
        p = mask_dir / f"{base_stem}__{roi}.png"
        if p.exists():
            m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            m = (m > 127).astype(np.uint8)
            if m.shape[:2] != shape[:2]:
                m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
        else:
            m = np.ones(shape[:2], np.uint8)
        masks[roi] = m
    return masks


In [ ]:
# 3) METRICS ───────────────────────────────────────────────────────────────────
def ssim_roi(x, y, mask):
    # grayscale SSIM on masked area
    xg = cv2.cvtColor((x*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    yg = cv2.cvtColor((y*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    m  = mask.astype(bool)
    if m.sum() == 0: return np.nan
    xw, yw = xg[m], yg[m]
    C1, C2 = 0.01**2, 0.03**2
    mx, my = xw.mean(), yw.mean()
    vx, vy = xw.var(),  yw.var()
    cov = ((xw-mx)*(yw-my)).mean()
    return float(((2*mx*my+C1)*(2*cov+C2))/((mx**2+my**2+C1)*(vx+vy+C2)))

if USE_TORCH:
    _lpips = piq.LPIPS(reduction='mean')
    def ms_ssim_piq(x,y):
        X = torch.from_numpy(x).permute(2,0,1).unsqueeze(0)
        Y = torch.from_numpy(y).permute(2,0,1).unsqueeze(0)
        with torch.no_grad(): return float(piq.multi_scale_ssim(X,Y,data_range=1.0).item())
    def fsim_piq(x,y):
        X = torch.from_numpy(x).permute(2,0,1).unsqueeze(0)
        Y = torch.from_numpy(y).permute(2,0,1).unsqueeze(0)
        with torch.no_grad(): return float(piq.fsim(X,Y,data_range=1.0).item())
    def lpips_piq(x,y):
        X = torch.from_numpy(x).permute(2,0,1).unsqueeze(0)
        Y = torch.from_numpy(y).permute(2,0,1).unsqueeze(0)
        with torch.no_grad(): return float(_lpips(X,Y).item())
else:
    def ms_ssim_piq(x,y): return np.nan
    def fsim_piq(x,y):    return np.nan
    def lpips_piq(x,y):   return np.nan

def edge_overlap(a,b, sigma=1.8):
    ea = canny(cv2.cvtColor((a*255).astype(np.uint8), cv2.COLOR_RGB2GRAY), sigma=sigma)
    eb = canny(cv2.cvtColor((b*255).astype(np.uint8), cv2.COLOR_RGB2GRAY), sigma=sigma)
    return float(np.logical_and(ea,eb).sum() / (np.logical_or(ea,eb).sum()+1e-9))

def bhattacharyya(h1,h2): return -np.log(np.sqrt(h1*h2).sum()+1e-12)
def hist_distance(rgb_a, rgb_b, bins=64):
    ha = cv2.calcHist([ (rgb_a*255).astype(np.uint8) ], [0,1,2], None, [bins]*3, [0,256]*3).flatten()
    hb = cv2.calcHist([ (rgb_b*255).astype(np.uint8) ], [0,1,2], None, [bins]*3, [0,256]*3).flatten()
    ha /= (ha.sum()+1e-9); hb /= (hb.sum()+1e-9)
    return float(bhattacharyya(ha,hb))

def dice_iou(ma, mb):
    inter = (ma & mb).sum(); union = (ma | mb).sum()
    dice = 2*inter/(ma.sum()+mb.sum()+1e-9); iou = inter/(union+1e-9)
    return float(dice), float(iou)


In [ ]:
# 4) ALIGNMENT ─────────────────────────────────────────────────────────────────
def ecc_align_ref_to_cand(ref_rgb, cand_rgb, mode=cv2.MOTION_AFFINE,
                          n_iter=300, eps=1e-4):
    """
    Warp the REFERENCE to the candidate using ECC (affine).
    Returns: ref_aligned(float01), used_alignment(bool).
    """
    ch, cw = cand_rgb.shape[:2]
    ref_r  = cv2.resize((ref_rgb*255).astype(np.uint8), (cw, ch), interpolation=cv2.INTER_AREA)
    ref_g  = cv2.cvtColor(ref_r,  cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    cand_g = cv2.cvtColor((cand_rgb*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0

    warp_matrix = np.eye(2, 3, dtype=np.float32) if mode!=cv2.MOTION_HOMOGRAPHY else np.eye(3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, n_iter, eps)
    try:
        ref_blur, cand_blur = cv2.GaussianBlur(ref_g,(5,5),0), cv2.GaussianBlur(cand_g,(5,5),0)
        _, warp_matrix = cv2.findTransformECC(cand_blur, ref_blur, warp_matrix, mode, criteria, None, 5)
        if mode == cv2.MOTION_HOMOGRAPHY:
            ref_warp = cv2.warpPerspective(ref_r, warp_matrix, (cw, ch), flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
        else:
            ref_warp = cv2.warpAffine(ref_r, warp_matrix, (cw, ch), flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
        return to_float01(ref_warp), True
    except cv2.error:
        return to_float01(ref_r), False

# (Optional) ORB stats for reporting (not used for metrics)
def orb_stats(src_rgb, ref_rgb, max_w=2200):
    sgray = cv2.cvtColor(src_rgb, cv2.COLOR_RGB2GRAY)
    rgray = cv2.cvtColor(ref_rgb, cv2.COLOR_RGB2GRAY)
    def _down(x):
        s=max(x.shape); 
        if s>max_w: r=max_w/s; return cv2.resize(x,None,fx=r,fy=r), r
        return x,1.0
    sgray, rs = _down(sgray); rgray, rr = _down(rgray)
    orb=cv2.ORB_create(5000); kp1,des1=orb.detectAndCompute(sgray,None); kp2,des2=orb.detectAndCompute(rgray,None)
    if des1 is None or des2 is None or len(kp1)<10 or len(kp2)<10: return 0, 0.0
    bf=cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    good=[]
    for m,n in bf.knnMatch(des1,des2,k=2):
        if m.distance < 0.75*n.distance: good.append(m)
    if len(good)<8: return len(good), 0.0
    src_pts=np.float32([kp1[m.queryIdx].pt for m in good])/rs
    ref_pts=np.float32([kp2[m.trainIdx].pt for m in good])/rr
    H,mask=cv2.findHomography(src_pts, ref_pts, cv2.RANSAC, 5.0)
    inlier=float(mask.sum())/len(mask) if (mask is not None and len(mask)>0) else 0.0
    return len(good), inlier


In [ ]:
# 5) DISCOVER FINALISTS ────────────────────────────────────────────────────────
def find_candidates(root=CAND_DIR):
    pats=("*.png","*.jpg","*.jpeg","*.tif","*.bmp","*/*.png","*/*.jpg","*/*.jpeg")
    files=[]
    for pat in pats: files += glob.glob(str(root/pat))
    files = [Path(p) for p in files if "masks" not in Path(p).parts]
    return sorted(set(files))

cands = find_candidates(CAND_DIR)
print(f"Found {len(cands)} finalist candidate(s).")
assert len(cands) > 0, "No images found under 'images/'."


In [ ]:
# 6) METRICS WITH ECC (reference→candidate) ───────────────────────────────────
ref_rgb = to_float01(imread_rgb(REF_PATH))
roi_names = ["statue","drapery","hair","hands","niche","silhouette"]

rows=[]
for p in cands:
    stem = p.stem
    cand = to_float01(imread_rgb(p))

    # Align REFERENCE to candidate (affine ECC). Candidates remain raw.
    A, ok_ecc = ecc_align_ref_to_cand(ref_rgb, cand, mode=cv2.MOTION_AFFINE)
    B = cand

    # Optional ORB stats (for reporting only)
    try:
        nmatch, inlier_rt = orb_stats((cand*255).astype(np.uint8), (ref_rgb*255).astype(np.uint8))
    except Exception:
        nmatch, inlier_rt = 0, 0.0

    # ROI masks (auto-resized)
    masks = load_masks(stem, roi_names, A.shape)

    # ROI-weighted SSIM
    vals, wts = [], []
    for rname in roi_names:
        m = masks[rname]
        vals.append(ssim_roi(A, B, m))
        wts.append(m.sum())
    wts = np.array(wts, np.float32); wts = wts/wts.sum() if wts.sum()>0 else np.ones_like(wts)/len(wts)
    ssim_w = float(np.sum(wts*np.array(vals)))

    # Additional metrics
    mse = float(np.mean((A - B)**2))
    mae = float(np.mean(np.abs(A - B)))

    rows.append(dict(
        image=p.name, path=str(p),
        alignment=("ECC_affine" if ok_ecc else "resize_only"),
        matches=int(nmatch), inlier_ratio=float(inlier_rt),
        SSIM_roi=ssim_w,
        MS_SSIM=ms_ssim_piq(A, B),
        FSIM=fsim_piq(A, B),
        LPIPS=lpips_piq(A, B),
        PSNR=peak_signal_noise_ratio(A, B, data_range=1.0),
        EDGE_OVERLAP=edge_overlap(A, B),
        HIST_BHATT=hist_distance(A, B),
        MSE=mse, MAE=mae,
        DICE=np.nan, IOU=np.nan
    ))

df = pd.DataFrame(rows)
df.to_csv(OUT/"tables/finalists_metrics.csv", index=False)
print("Saved:", OUT/"tables/finalists_metrics.csv")
df.head()


In [ ]:
# 7) RANKING ─ safe normalization that skips all-NaN metrics
pos_all = ["SSIM_roi","MS_SSIM","FSIM","PSNR","EDGE_OVERLAP","DICE","IOU"]
neg_all = ["LPIPS","HIST_BHATT","MSE","MAE"]

# keep only metrics that have at least one finite value
pos = [m for m in pos_all if np.isfinite(df[m].to_numpy()).any()] 
neg = [m for m in neg_all if np.isfinite(df[m].to_numpy()).any()]

dfN = df.copy()
for m in pos + neg:
    v = dfN[m].astype(float)
    vmin = np.nanmin(v) if np.isfinite(v).any() else np.nan
    vmax = np.nanmax(v) if np.isfinite(v).any() else np.nan
    if not np.isfinite(vmin) or not np.isfinite(vmax) or (vmax - vmin) < 1e-12:
        dfN[m + "_n"] = np.nan
    else:
        n = (v - vmin) / (vmax - vmin)
        if m in neg: n = 1.0 - n
        dfN[m + "_n"] = n


# Composite weights (sum to 1 over available components)
weights = dict(
    MS_SSIM_n      = 0.28,
    LPIPS_n        = 0.28,
    EDGE_OVERLAP_n = 0.14,
    PSNR_n         = 0.10,
    FSIM_n         = 0.10,
    MSE_n          = 0.05,
    MAE_n          = 0.05
)

def composite_score(row):
    comps = {k: row.get(k, np.nan) for k in weights.keys()}
    w_eff = {k:w for k,w in weights.items() if not np.isnan(comps[k])}
    if not w_eff: return np.nan
    s = sum(w_eff.values()); w_eff = {k:v/s for k,v in w_eff.items()}
    return sum(w_eff[k]*comps[k] for k in w_eff)

dfN["Score"] = dfN.apply(composite_score, axis=1)
df_ranked = dfN.sort_values("Score", ascending=False).reset_index(drop=True)
df_ranked.to_csv(OUT/"tables/finalists_ranked.csv", index=False)
print("Saved:", OUT/"tables/finalists_ranked.csv")
display(df_ranked[["image","alignment","Score","SSIM_roi","MS_SSIM","LPIPS","EDGE_OVERLAP","PSNR"]].head(TOP_K))

# Export Top-K and bar chart
import shutil
export_dir = OUT/"top_k"; export_dir.mkdir(exist_ok=True)
topK = df_ranked.head(TOP_K).copy()
for _,r in topK.iterrows():
    src = Path(r["path"]); dst = export_dir/src.name
    if not dst.exists(): shutil.copy2(src, dst)

plt.figure(figsize=(8,4))
plt.bar(topK["image"], topK["Score"])
plt.xticks(rotation=45, ha='right'); plt.ylabel("Composite score (0–1)")
plt.title(f"Top-{TOP_K} finalists by composite score")
plt.tight_layout(); plt.savefig(OUT/"figs/topk_bar.png", dpi=200); plt.close()
print("Saved:", OUT/"figs/topk_bar.png")


In [ ]:
# 8) HEATMAPS ──────────────────────────────────────────────────────────────────
ref_rgb = to_float01(imread_rgb(REF_PATH))
saved = []
for _, r in topK.iterrows():
    cand_path = Path(r["path"])
    cand_rgb  = to_float01(imread_rgb(cand_path))

    ref_to_cand, ok = ecc_align_ref_to_cand(ref_rgb, cand_rgb, mode=cv2.MOTION_AFFINE)

    ag = cv2.cvtColor((ref_to_cand*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    bg = cv2.cvtColor((cand_rgb*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    err = np.abs(ag - bg)

    plt.figure(figsize=(15,4))
    plt.subplot(1,3,1); plt.imshow(ref_rgb);  plt.axis('off'); plt.title("Reference")
    plt.subplot(1,3,2); plt.imshow(cand_rgb); plt.axis('off'); plt.title(f"{r['image']} (raw)")
    plt.subplot(1,3,3); im=plt.imshow(err, cmap='inferno', vmin=0, vmax=1)
    plt.axis('off'); plt.title("Abs error (ref→cand aligned)" if ok else "Abs error (ref resized)")
    plt.colorbar(im, fraction=.046, pad=.04)
    plt.tight_layout()
    outp = OUT / f"figs/heatmap_{Path(r['image']).stem}.png"
    plt.savefig(outp, dpi=220); plt.close()
    saved.append(outp)

print(f"Saved {len(saved)} heatmap(s) to:", OUT/"figs")
from IPython.display import Image, display
if saved: display(Image(filename=str(saved[0])))


In [ ]:
# 8) UNCERTAINTY (bootstrap) ──────────────────────────────────────────────────
def bootstrap_ci(x, B=5000, alpha=0.05):
    x=np.array(x, float); x=x[~np.isnan(x)]
    if len(x)==0: return (np.nan,np.nan,np.nan)
    rng=np.random.default_rng(42)
    boots=np.array([rng.choice(x, size=len(x), replace=True).mean() for _ in range(B)])
    lo,hi=np.percentile(boots,[100*alpha/2,100*(1-alpha/2)])
    return (x.mean(), lo, hi)

# CIs for core metrics across all finalists
metrics_core = ["SSIM_roi","MS_SSIM","LPIPS","EDGE_OVERLAP","PSNR","FSIM","HIST_BHATT"]
rows_ci=[]
for m in metrics_core:
    mu,lo,hi = bootstrap_ci(df[m].values)
    rows_ci.append(dict(metric=m, mean=mu, ci_lo=lo, ci_hi=hi))
pd.DataFrame(rows_ci).to_csv(OUT/"tables/finalist_metric_CIs.csv", index=False)

# CI for composite score among top-K
mu,lo,hi = bootstrap_ci(topK["Score"].values)
pd.DataFrame([dict(stat="TopK mean Score", mean=mu, ci_lo=lo, ci_hi=hi)]).to_csv(OUT/"tables/topk_score_CI.csv", index=False)
print(f"Top-{TOP_K} mean Score = {mu:.3f} (95% CI {lo:.3f}–{hi:.3f})")


In [ ]:
# 10) EXPERT FUSION (Optional) ────────────────────────────────────────────────
# If you have expert ratings per candidate, share a screenshot of columns;
# I’ll adapt a reader to compute ICC/α/κ and add EXPERTS_n into the ranking.
EXPERTS_PATH = DATA_ROOT/"experts.xlsx"
if EXPERTS_PATH.exists():
    print("experts.xlsx found. If you want fusion, show me the first 10 columns/rows and I'll wire it in.")
else:
    print("No experts.xlsx detected — skipping fusion.")


In [ ]:
# 11) REPRODUCIBILITY CAPSULE ─────────────────────────────────────────────────
capsule = dict(
    python=tuple(map(int, __import__('sys').version.split()[0].split("."))),
    numpy=np.__version__,
    cv2=cv2.__version__,
    skimage=__import__('skimage').__version__,
    pandas=pd.__version__,
    torch=(torch.__version__ if USE_TORCH else "NA"),
    piq=(piq.__version__ if USE_TORCH else "NA"),
    seeds=dict(numpy=42, torch=(int(torch.initial_seed()) if USE_TORCH else None)),
    paths=dict(REF=str(REF_PATH), CAND=str(CAND_DIR), MASK=str(MASK_DIR), OUT=str(OUT))
)
(OUT/"reproducibility_capsule.json").write_text(json.dumps(capsule, indent=2))
print("Saved:", OUT/"reproducibility_capsule.json")


In [ ]:
# 12) QUICK SUMMARY ───────────────────────────────────────────────────────────
print("\nTop candidates:")
display(df_ranked[["image","alignment","Score","SSIM_roi","MS_SSIM","LPIPS","EDGE_OVERLAP","PSNR"]].head(TOP_K))
print("\nOutputs written to:", OUT.resolve())


In [ ]:
# A1) What’s already saved
from pathlib import Path
from IPython.display import Image, display
from glob import glob

fig_dir = OUT / "figs"
print("Saved plots in:", fig_dir.resolve())
for p in sorted(fig_dir.glob("*.png")):
    print("-", p.name)

# A2) Show a quick gallery of heatmaps + the top-k bar
gallery = sorted(glob(str(fig_dir / "heatmap_*.png")))[:6]  # first 6 to avoid flooding
for g in gallery:
    display(Image(filename=g))
display(Image(filename=str(fig_dir / "topk_bar.png")))


In [ ]:
# B) EXTRA METRICS: GMSD (↓ better) and VSI (↑ better) for completeness
if USE_TORCH:
    import torch, piq
    def gmsd_piq(x,y):
        X = torch.from_numpy(x).permute(2,0,1).unsqueeze(0)
        Y = torch.from_numpy(y).permute(2,0,1).unsqueeze(0)
        with torch.no_grad(): 
            return float(piq.gmsd(X,Y, data_range=1.0).item())
    def vsi_piq(x,y):
        X = torch.from_numpy(x).permute(2,0,1).unsqueeze(0)
        Y = torch.from_numpy(y).permute(2,0,1).unsqueeze(0)
        with torch.no_grad(): 
            return float(piq.vsi(X,Y, data_range=1.0).item())
else:
    def gmsd_piq(x,y): return np.nan
    def vsi_piq(x,y):  return np.nan

# Compute for all rows using the same ECC alignment you already rely on
ref_rgb = to_float01(imread_rgb(REF_PATH))
g_rows = []
for _, r in df.iterrows():
    cand = to_float01(imread_rgb(Path(r["path"])))
    A, _ = ecc_align_ref_to_cand(ref_rgb, cand, mode=cv2.MOTION_AFFINE)
    B = cand
    g_rows.append(dict(image=r["image"], GMSD=gmsd_piq(A,B), VSI=vsi_piq(A,B)))

df_extra = pd.DataFrame(g_rows)
df = df.merge(df_extra, on="image", how="left")
df.to_csv(OUT/"tables/finalists_metrics.csv", index=False)
print("Updated with extra metrics:", OUT/"tables/finalists_metrics.csv")

# Re-normalize & re-rank including GMSD(neg) and VSI(pos)
pos_add = ["VSI"]
neg_add = ["GMSD"]

pos_all = ["SSIM_roi","MS_SSIM","FSIM","PSNR","EDGE_OVERLAP","DICE","IOU","VSI"]
neg_all = ["LPIPS","HIST_BHATT","MSE","MAE","GMSD"]
pos = [m for m in pos_all if m in df and np.isfinite(df[m]).any()]
neg = [m for m in neg_all if m in df and np.isfinite(df[m]).any()]

dfN = df.copy()
for m in pos + neg:
    v = dfN[m].astype(float)
    if not np.isfinite(v).any() or (np.nanmax(v)-np.nanmin(v)) < 1e-12:
        dfN[m+"_n"] = np.nan
    else:
        n = (v - np.nanmin(v))/(np.nanmax(v)-np.nanmin(v))
        if m in neg: n = 1.0 - n
        dfN[m+"_n"] = n

# Slightly adjust weights to give small room for the new metrics
weights = dict(
    MS_SSIM_n      = 0.24,
    LPIPS_n        = 0.24,
    EDGE_OVERLAP_n = 0.12,
    PSNR_n         = 0.09,
    FSIM_n         = 0.09,
    MSE_n          = 0.05,
    MAE_n          = 0.05,
    VSI_n          = 0.06,  # new
    GMSD_n         = 0.06   # new
)

def composite_score(row):
    comps = {k: row.get(k, np.nan) for k in weights}
    w_eff = {k:w for k,w in weights.items() if np.isfinite(comps[k])}
    if not w_eff: return np.nan
    s = sum(w_eff.values()); w_eff = {k:v/s for k,v in w_eff.items()}
    return sum(w_eff[k]*comps[k] for k in w_eff)

dfN["Score"] = dfN.apply(composite_score, axis=1)
df_ranked = dfN.sort_values("Score", ascending=False).reset_index(drop=True)
df_ranked.to_csv(OUT/"tables/finalists_ranked.csv", index=False)
display(df_ranked[["image","Score","SSIM_roi","MS_SSIM","LPIPS","EDGE_OVERLAP","PSNR","VSI","GMSD"]].head(TOP_K))

# refresh Top-K bar
plt.figure(figsize=(8,4))
plt.bar(df_ranked.head(TOP_K)["image"], df_ranked.head(TOP_K)["Score"])
plt.xticks(rotation=45, ha='right'); plt.ylabel("Composite score (0–1)")
plt.title(f"Top-{TOP_K} finalists by composite score (with VSI/GMSD)")
plt.tight_layout(); plt.savefig(OUT/"figs/topk_bar.png", dpi=200); plt.close()
display(Image(filename=str(OUT/"figs/topk_bar.png")))


In [ ]:
import seaborn as sns  # if not available: pip install seaborn
corr_cols = ["SSIM_roi","MS_SSIM","FSIM","PSNR","EDGE_OVERLAP","LPIPS","HIST_BHATT","MSE","MAE","VSI","GMSD"]
corr_cols = [c for c in corr_cols if c in df]
corr = df[corr_cols].corr(method="spearman")
plt.figure(figsize=(8,6))
sns.heatmap(corr, vmin=-1, vmax=1, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Spearman correlation among metrics")
plt.tight_layout(); plt.savefig(OUT/"figs/metric_correlation.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/metric_correlation.png")))


In [ ]:
# Build absolute-error CDFs for top-K
ref_rgb = to_float01(imread_rgb(REF_PATH))
plt.figure(figsize=(7,5))
for _, r in df_ranked.head(TOP_K).iterrows():
    cand = to_float01(imread_rgb(Path(r["path"])))
    ref_to_cand, _ = ecc_align_ref_to_cand(ref_rgb, cand, mode=cv2.MOTION_AFFINE)
    ag = cv2.cvtColor((ref_to_cand*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    bg = cv2.cvtColor((cand*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    err = np.abs(ag - bg).ravel()
    xs = np.sort(err); cdf = np.arange(1,len(xs)+1)/len(xs)
    plt.plot(xs, cdf, label=r["image"])
plt.xlabel("Absolute grayscale error"); plt.ylabel("CDF")
plt.title("Per-image error CDF (top-K)")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig(OUT/"figs/error_cdf_topk.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/error_cdf_topk.png")))


In [ ]:
def bland_altman(a, b, nameA, nameB, outname):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m  = (a + b)/2
    d  = a - b
    md = np.nanmean(d); sd = np.nanstd(d, ddof=1)
    lo, hi = md - 1.96*sd, md + 1.96*sd
    plt.figure(figsize=(6,4))
    plt.scatter(m, d, s=18, alpha=.7)
    plt.axhline(md, color='k'); plt.axhline(lo, color='r', ls='--'); plt.axhline(hi, color='r', ls='--')
    plt.xlabel(f"Mean of {nameA} and {nameB}"); plt.ylabel(f"Difference ({nameA}-{nameB})")
    plt.title(f"Bland–Altman: {nameA} vs {nameB}")
    plt.tight_layout(); plt.savefig(OUT/f"figs/{outname}.png", dpi=220); plt.close()
    display(Image(filename=str(OUT/f"figs/{outname}.png")))

# Example: MS-SSIM vs VSI (both ↑)
if {"MS_SSIM","VSI"}.issubset(df.columns):
    bland_altman(df["MS_SSIM"], df["VSI"], "MS_SSIM", "VSI", "ba_ms_ssim_vs_vsi")
# Example: LPIPS (↓) vs GMSD (↓) — invert LPIPS to compare as ↑
if {"LPIPS","GMSD"}.issubset(df.columns):
    bland_altman(1-df["LPIPS"], 1-df["GMSD"], "1-LPIPS", "1-GMSD", "ba_inv_lpips_vs_inv_gmsd")


In [ ]:
# Sweep ±20% around each weight (re-normalize others) and see score change for the winner
base = df_ranked.iloc[0]["image"]
base_score = df_ranked.iloc[0]["Score"]

sens = []
for k in list(weights.keys()):
    for s in (-0.2, +0.2):
        w_new = weights.copy()
        w_new[k] = max(w_new[k]*(1+s), 0)
        # re-normalize to sum=1
        S = sum(w_new.values()); w_new = {kk:vv/S for kk,vv in w_new.items()}
        tmp = dfN.copy()
        tmp["Score_tmp"] = tmp.apply(lambda row: sum(w_new[kk]*row.get(kk, np.nan) 
                                                    for kk in w_new if np.isfinite(row.get(kk, np.nan))), axis=1)
        new_top = tmp.sort_values("Score_tmp", ascending=False).iloc[0]
        sens.append(dict(metric=k, shift=f"{'+' if s>0 else ''}{int(s*100)}%", 
                         top_image=new_top["image"], delta=(float(new_top["Score_tmp"])-float(base_score))))
sens_df = pd.DataFrame(sens)
sens_df.to_csv(OUT/"tables/weight_sensitivity.csv", index=False)
display(sens_df)

# Tornado (absolute |delta| by metric, worst case of ±20%)
worst = sens_df.pivot_table(index="metric", values="delta", aggfunc=lambda x: np.max(np.abs(x))).sort_values("delta")
plt.figure(figsize=(6,5))
plt.barh(worst.index, worst["delta"])
plt.xlabel("Max |ΔScore| over ±20% weight change")
plt.title("Composite score sensitivity")
plt.tight_layout(); plt.savefig(OUT/"figs/tornado_weight_sensitivity.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/tornado_weight_sensitivity.png")))


In [ ]:
# D) TOST equivalence on Score (top-1 vs top-2) with epsilon
eps = 0.02  # practical equivalence margin on the 0–1 score scale
if len(df_ranked)>=2:
    x = df_ranked.iloc[0]["Score"]; y = df_ranked.iloc[1]["Score"]
    d = np.array([x - y])
    # one-sample TOST: H0: |μ| >= eps ; HA: |μ| < eps
    # Using normal approx with bootstrap variance
    boots = np.array([np.mean(np.random.choice(d, size=len(d), replace=True)) for _ in range(5000)])
    mu = boots.mean(); sd = boots.std(ddof=1)
    z_lo = (mu + eps) / (sd+1e-12); z_hi = (mu - eps) / (sd+1e-12)
    from scipy.stats import norm
    p1 = 1 - norm.cdf(z_lo); p2 = norm.cdf(z_hi)
    p_equiv = max(p1, p2)  # both must be significant; report conservative
    print(f"TOST top1 vs top2: Δ={float(x-y):.3f}, eps={eps}, p_equiv={p_equiv:.3g} (smaller is better)")


In [ ]:
# === 9A) ADVANCED METRICS: Deep-Feature Similarity (DFS) + Composition/Structure Similarity (CSS)

import torch, torchvision
import numpy as np
import cv2
from pathlib import Path

USE_CUDA = torch.cuda.is_available()

# ---- Deep feature extractor (VGG16 by default; switch to resnet50 if you prefer) ----
class _FeatHook(torch.nn.Module):
    def __init__(self, backbone='vgg16'):
        super().__init__()
        if backbone == 'vgg16':
            weights = torchvision.models.VGG16_Weights.DEFAULT
            m = torchvision.models.vgg16(weights=weights).features
            self.backbone = m.eval()
            self.pick = [3, 8, 15, 22]  # conv blocks
            self.out_channels = 64+128+256+512
        elif backbone == 'resnet50':
            weights = torchvision.models.ResNet50_Weights.DEFAULT
            m = torchvision.models.resnet50(weights=weights)
            # use layers up to layer3 for mid-level semantics
            self.backbone = torch.nn.Sequential(
                m.conv1, m.bn1, m.relu, m.maxpool,
                m.layer1, m.layer2, m.layer3
            ).eval()
            self.pick = None
            self.out_channels = 1024
        else:
            raise ValueError("backbone must be 'vgg16' or 'resnet50'")
        for p in self.backbone.parameters(): p.requires_grad_(False)
        if USE_CUDA: self.backbone.cuda()

    def _pre(self, x_np):
        x = torch.from_numpy(x_np).permute(2,0,1).unsqueeze(0).float()  # 1x3xHxW
        if x.max() <= 1.0: x = x*255.0
        # VGG normalization
        mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
        std  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
        x = x/255.0
        x = (x-mean)/std
        if USE_CUDA: x = x.cuda()
        return x

    def forward(self, img_np):
        x = self._pre(img_np)
        if self.pick is None:
            y = self.backbone(x)               # resnet feature map
            g = torch.nn.functional.adaptive_avg_pool2d(y, 1).flatten(1)
            return g.detach().cpu().numpy().ravel()
        else:
            feats = []
            cur = x
            for i,layer in enumerate(self.backbone):
                cur = layer(cur)
                if i in self.pick:
                    feats.append(torch.nn.functional.adaptive_avg_pool2d(cur, 1))
            g = torch.cat(feats, dim=1).flatten(1)
            return g.detach().cpu().numpy().ravel()

def deep_feature_similarity(A, B, model=None):
    """Cosine similarity of pooled deep features in [0,1]."""
    if model is None:
        model = _FeatHook('vgg16')
    fa = model(A); fb = model(B)
    num = (fa*fb).sum()
    den = (np.linalg.norm(fa)*np.linalg.norm(fb) + 1e-9)
    cos = float(num/den)
    # map from [-1,1]→[0,1]
    return 0.5*(cos+1.0)

# ---- Composition / Structure Similarity (CSS) via ORB + Procrustes ----
def composition_structure_similarity(A, B, max_kp=1500):
    """1 - normalized Procrustes residual from matched keypoints (higher is better)."""
    gA = cv2.cvtColor((A*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    gB = cv2.cvtColor((B*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    orb = cv2.ORB_create(max_kp)
    k1,d1 = orb.detectAndCompute(gA,None)
    k2,d2 = orb.detectAndCompute(gB,None)
    if d1 is None or d2 is None or len(k1)<8 or len(k2)<8: 
        return np.nan
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    good=[]
    for m,n in bf.knnMatch(d1,d2,k=2):
        if m.distance < 0.75*n.distance: good.append(m)
    if len(good) < 12: return np.nan
    P = np.float32([k1[m.queryIdx].pt for m in good])
    Q = np.float32([k2[m.trainIdx].pt for m in good])

    # Procrustes (translation/scale/rotation) on centered points
    P0 = P - P.mean(0); Q0 = Q - Q.mean(0)
    sP = np.linalg.norm(P0) + 1e-9; sQ = np.linalg.norm(Q0) + 1e-9
    Pn, Qn = P0/sP, Q0/sQ
    U,S,Vt = np.linalg.svd(Qn.T @ Pn)
    R = U @ Vt
    PnR = Pn @ R.T
    resid = np.sqrt(((PnR - Qn)**2).sum(axis=1).mean())  # RMS
    # Normalize to [0,1]: 0 residual → 1.0; higher residual → lower similarity
    css = float(np.clip(1.0 - resid, 0.0, 1.0))
    return css

# ---- Compute & merge into df using same ECC alignment A (ref→cand), B=cand ----
ref_rgb = to_float01(imread_rgb(REF_PATH))
adv_rows = []
_model = _FeatHook('vgg16')  # load once

for _, r in df.iterrows():
    cand = to_float01(imread_rgb(Path(r["path"])))
    A, _ = ecc_align_ref_to_cand(ref_rgb, cand, mode=cv2.MOTION_AFFINE)
    B = cand
    dfs = deep_feature_similarity(A,B, _model)    # ↑ better
    css = composition_structure_similarity(A,B)   # ↑ better
    adv_rows.append(dict(image=r["image"], DFS=dfs, CSS=css))

df_adv = pd.DataFrame(adv_rows)
df = df.merge(df_adv, on="image", how="left")
df.to_csv(OUT/"tables/finalists_metrics.csv", index=False)
print("Added advanced metrics: DFS, CSS")
df.tail(3)


In [ ]:
# === 9B) RE-RANK with advanced metrics + multiple scenarios

import numpy as np
import pandas as pd
from collections import OrderedDict

# Normalize (safe) including new metrics
pos_all = ["SSIM_roi","MS_SSIM","FSIM","PSNR","EDGE_OVERLAP","DICE","IOU","VSI","CSS","DFS"]
neg_all = ["LPIPS","HIST_BHATT","MSE","MAE","GMSD"]

dfN = df.copy()
for m in pos_all + neg_all:
    if m not in dfN.columns: 
        dfN[m+"_n"] = np.nan
        continue
    v = dfN[m].astype(float)
    if not np.isfinite(v).any() or (np.nanmax(v)-np.nanmin(v))<1e-12:
        dfN[m+"_n"] = np.nan
    else:
        n = (v - np.nanmin(v)) / (np.nanmax(v)-np.nanmin(v))
        if m in neg_all: n = 1.0 - n
        dfN[m+"_n"] = n

def rerank(weights):
    # renormalize to available components per row
    def score_row(row):
        comps = {k: row.get(k, np.nan) for k in weights}
        w_eff = {k:w for k,w in weights.items() if np.isfinite(comps[k])}
        if not w_eff: return np.nan
        s = sum(w_eff.values()); w_eff = {k:v/s for k,v in w_eff.items()}
        return sum(w_eff[k]*comps[k] for k in w_eff)
    d = dfN.copy()
    d["Score"] = d.apply(score_row, axis=1)
    return d.sort_values("Score", ascending=False).reset_index(drop=True)

# Baseline: your last mixed weights (adapted to include DFS/CSS with small mass)
W_base = OrderedDict({
    "MS_SSIM_n":0.20, "LPIPS_n":0.20, "EDGE_OVERLAP_n":0.12,
    "PSNR_n":0.09, "FSIM_n":0.09, "MSE_n":0.05, "MAE_n":0.05,
    "VSI_n":0.06, "GMSD_n":0.06, "CSS_n":0.04, "DFS_n":0.04
})

# Scenario weights (sum will renormalize in scoring)
W_struct   = OrderedDict({"CSS_n":0.30,"EDGE_OVERLAP_n":0.20,"MS_SSIM_n":0.20,"PSNR_n":0.10,"FSIM_n":0.10,"LPIPS_n":0.10})
W_percept  = OrderedDict({"DFS_n":0.35,"LPIPS_n":0.25,"MS_SSIM_n":0.20,"VSI_n":0.10,"FSIM_n":0.10})
W_ssim     = OrderedDict({"MS_SSIM_n":0.50,"SSIM_roi_n":0.20,"LPIPS_n":0.15,"PSNR_n":0.10,"EDGE_OVERLAP_n":0.05})
W_expert   = OrderedDict({"SSIM_roi_n":0.40,"LPIPS_n":0.30,"FSIM_n":0.10,"MS_SSIM_n":0.10,"VSI_n":0.10})

scenarios = OrderedDict([
    ("BASE",   W_base),
    ("STRUCT", W_struct),
    ("PERCEPT",W_percept),
    ("SSIM+",  W_ssim),
    ("EXPERT", W_expert),
])

ranked_scenarios = {name: rerank(W) for name,W in scenarios.items()}

# Save and preview top-8 under each scenario
tbl = []
for name, d in ranked_scenarios.items():
    top = d.head(8)[["image","Score"]].copy()
    top["scenario"] = name
    tbl.append(top)
pd.concat(tbl).to_csv(OUT/"tables/rerank_scenarios_top8.csv", index=False)

for name in scenarios.keys():
    print("\n---", name, "---")
    display(ranked_scenarios[name].head(8)[["image","Score","CSS","DFS","MS_SSIM","LPIPS","EDGE_OVERLAP","PSNR"]])


In [ ]:
# === 9C) EXPERT OVERLAY & RANK OVERLAP ===
expert_list = {"0.png","4.png","6.png","9.png","11.png","13.png","14.png","15.png"}  # your set

import matplotlib.pyplot as plt

def overlap_curve(d, expert_set, Ks=(3,5,8,10)):
    imgs = list(d["image"])
    res = []
    for K in Ks:
        res.append(len(set(imgs[:K]) & expert_set))
    return Ks, res

Ks = (3,5,8,10)
plt.figure(figsize=(6,4))
for name, d in ranked_scenarios.items():
    x,y = overlap_curve(d, expert_list, Ks)
    plt.plot(x,y, marker='o', label=name)
plt.xlabel("Top-K"); plt.ylabel("Intersection with expert shortlist")
plt.title("Expert–algorithm rank overlap")
plt.xticks(Ks); plt.legend()
plt.tight_layout(); plt.savefig(OUT/"figs/rank_overlap_expert.png", dpi=220); plt.close()

from IPython.display import Image, display
display(Image(filename=str(OUT/"figs/rank_overlap_expert.png")))


In [ ]:
# === 9D) METRIC CORRELATION (Spearman) ===
try:
    import seaborn as sns
except ImportError:
    !pip -q install seaborn
    import seaborn as sns

corr_cols = [c for c in ["SSIM_roi","MS_SSIM","FSIM","PSNR","EDGE_OVERLAP","LPIPS","HIST_BHATT","MSE","MAE","VSI","GMSD","CSS","DFS"] if c in df.columns]
corr = df[corr_cols].corr(method="spearman")
plt.figure(figsize=(9,7))
sns.heatmap(corr, vmin=-1, vmax=1, cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Spearman correlation among metrics")
plt.tight_layout(); plt.savefig(OUT/"figs/metric_correlation.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/metric_correlation.png")))


In [ ]:
# === 9E) ROI HEATMAPS FOR TOP-K ===
roi_names = ["statue","drapery","hair","hands","niche","silhouette"]  # align to your masks logic

ref_rgb = to_float01(imread_rgb(REF_PATH))
for _, r in ranked_scenarios["EXPERT"].head(8).iterrows():  # show under expert-weight scenario
    cand = to_float01(imread_rgb(Path(r["path"])))
    ref_to_cand, _ = ecc_align_ref_to_cand(ref_rgb, cand, mode=cv2.MOTION_AFFINE)
    A = ref_to_cand; B = cand
    ag = cv2.cvtColor((A*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    bg = cv2.cvtColor((B*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    err = np.abs(ag-bg)

    masks = load_masks(Path(r["image"]).stem, roi_names, A.shape)
    plt.figure(figsize=(14,3))
    plt.subplot(1, len(roi_names)+1, 1); plt.imshow(B); plt.axis('off'); plt.title(r["image"])
    for i,roi in enumerate(roi_names, start=2):
        m = masks[roi].astype(bool)
        roi_err = np.zeros_like(err); roi_err[m] = err[m]
        plt.subplot(1, len(roi_names)+1, i); 
        im = plt.imshow(roi_err, cmap="inferno", vmin=0, vmax=1); plt.axis('off'); plt.title(roi)
    plt.tight_layout()
    plt.savefig(OUT/f"figs/roi_heat_{Path(r['image']).stem}.png", dpi=200); plt.close()


In [ ]:
# === 9F) RADAR CHARTS (per image) ===
def radar_plot(row, cols, title, outpath):
    vals = [float(row[c]) if np.isfinite(row[c]) else 0.0 for c in cols]
    angles = np.linspace(0, 2*np.pi, len(cols), endpoint=False).tolist()
    vals += vals[:1]; angles += angles[:1]
    fig=plt.figure(figsize=(4.5,4.5))
    ax=plt.subplot(111, polar=True)
    ax.plot(angles, vals, linewidth=2); ax.fill(angles, vals, alpha=.2)
    ax.set_thetagrids(np.degrees(angles[:-1]), cols)
    ax.set_title(title, y=1.08)
    ax.set_rlim(0,1)
    plt.tight_layout(); plt.savefig(outpath, dpi=220); plt.close()

radar_cols = [c for c in ["SSIM_roi_n","MS_SSIM_n","FSIM_n","PSNR_n","EDGE_OVERLAP_n","1-LPIPS_n","VSI_n","CSS_n","DFS_n"] if c in dfN.columns]
# quick helper: if "1-LPIPS_n" is not present, derive from LPIPS_n
if "1-LPIPS_n" not in dfN.columns and "LPIPS_n" in dfN.columns:
    dfN["1-LPIPS_n"] = dfN["LPIPS_n"]  # already inverted when normalized

for _, row in ranked_scenarios["EXPERT"].head(6).iterrows():
    radar_plot(row, radar_cols, f"Radar — {row['image']}", OUT/f"figs/radar_{Path(row['image']).stem}.png")

from glob import glob
for rp in sorted(glob(str(OUT/"figs/radar_*.png"))):
    display(Image(filename=rp))


In [ ]:
# === 9G) BOOTSTRAP VIOLIN/BOX PLOTS ===
def bootstrap_samples(x, B=4000):
    x = np.array(x, float); x = x[~np.isnan(x)]
    if len(x)==0: return np.array([])
    rng = np.random.default_rng(42)
    return np.array([rng.choice(x, size=len(x), replace=True).mean() for _ in range(B)])

metrics_show = [c for c in ["SSIM_roi","MS_SSIM","LPIPS","VSI","CSS","DFS","EDGE_OVERLAP","PSNR"] if c in df.columns]
samples = {m: bootstrap_samples(df[m].values) for m in metrics_show}

plt.figure(figsize=(9,5))
sns.violinplot(data=[samples[m] for m in metrics_show], inner="box")
plt.xticks(range(len(metrics_show)), metrics_show, rotation=30, ha='right')
plt.ylabel("Bootstrap mean distribution")
plt.title("Uncertainty (bootstrap) across metrics")
plt.tight_layout(); plt.savefig(OUT/"figs/violin_uncertainty.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/violin_uncertainty.png")))


In [ ]:
# === 9H) TORNADO — sensitivity of winner to ±20% weight perturbations ===
base_rank = ranked_scenarios["EXPERT"]
base_img  = base_rank.iloc[0]["image"]
base_score= base_rank.iloc[0]["Score"]

W = scenarios["EXPERT"].copy()
sens = []
for k in list(W.keys()):
    for s in (-0.2, +0.2):
        Wp = W.copy()
        Wp[k] = max(Wp[k]*(1+s), 0)
        S = sum(Wp.values()); Wp = {kk:vv/S for kk,vv in Wp.items()}
        tmp = rerank(Wp)
        top = tmp.iloc[0]
        sens.append(dict(metric=k, shift=f"{'+' if s>0 else ''}{int(s*100)}%", 
                         top_image=top["image"], delta=float(top["Score"]-base_score)))
sens_df = pd.DataFrame(sens)
sens_df.to_csv(OUT/"tables/tornado_expert_weights.csv", index=False)
display(sens_df)

worst = sens_df.pivot_table(index="metric", values="delta", aggfunc=lambda x: np.max(np.abs(x))).sort_values("delta")
plt.figure(figsize=(6,5))
plt.barh(worst.index, worst["delta"])
plt.xlabel("Max |ΔScore| over ±20% weight change"); plt.title("Composite score sensitivity (EXPERT weights)")
plt.tight_layout(); plt.savefig(OUT/"figs/tornado_expert_weights.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/tornado_expert_weights.png")))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1) Present?
p_exists = Path("images/6.png").exists()
print("images/6.png exists:", p_exists)

# 2) Row in metrics?
metrics_csv = OUT/"tables/finalists_metrics.csv"
df_all = pd.read_csv(metrics_csv)
has_row = (df_all["image"]=="6.png").any()
print("Row for 6.png in metrics table:", has_row)

# 3) Show full row (raw metrics)
row6 = df_all.loc[df_all["image"]=="6.png"]
display(row6.T)  # raw values

# 4) Show alignment status & match stats
if not row6.empty:
    print("\nAlignment:", row6["alignment"].values[0], 
          "| matches:", int(row6["matches"].values[0]) if "matches" in row6 else "[NA]",
          "| inlier_ratio:", float(row6["inlier_ratio"].values[0]) if "inlier_ratio" in row6 else "[NA]")


In [ ]:
# Recreate the normalized table used in ranking (same metric lists you used)
pos_all = ["SSIM_roi","MS_SSIM","FSIM","PSNR","EDGE_OVERLAP","DICE","IOU","VSI","CSS","DFS"]
neg_all = ["LPIPS","HIST_BHATT","MSE","MAE","GMSD"]

dfN = df_all.copy()
for m in pos_all + neg_all:
    if m not in dfN.columns:
        dfN[m+"_n"] = np.nan
        continue
    v = dfN[m].astype(float)
    if not np.isfinite(v).any() or (np.nanmax(v)-np.nanmin(v))<1e-12:
        dfN[m+"_n"] = np.nan
    else:
        n = (v - np.nanmin(v))/(np.nanmax(v)-np.nanmin(v))
        if m in neg_all: n = 1.0 - n
        dfN[m+"_n"] = n

# Use the weight scenario currently used for your chart (adjust if different)
weights = dict(
    MS_SSIM_n=0.20, LPIPS_n=0.20, EDGE_OVERLAP_n=0.12,
    PSNR_n=0.09, FSIM_n=0.09, MSE_n=0.05, MAE_n=0.05,
    VSI_n=0.06, GMSD_n=0.06, CSS_n=0.04, DFS_n=0.04
)

def score_row(r, W):
    comps = {k: r.get(k, np.nan) for k in W}
    w_eff = {k:w for k,w in W.items() if np.isfinite(comps[k])}
    if not w_eff: return np.nan, {}
    s = sum(w_eff.values()); w_eff = {k:v/s for k,v in w_eff.items()}
    contrib = {k: w_eff[k]*comps[k] for k in w_eff}
    return sum(contrib.values()), contrib

# get top-1 row from the ranked CSV you plotted
ranked_csv = OUT/"tables/finalists_ranked.csv"
df_ranked = pd.read_csv(ranked_csv)
top1 = df_ranked.iloc[0]["image"]
r6 = dfN.loc[dfN["image"]=="6.png"].iloc[0]
rt = dfN.loc[dfN["image"]==top1].iloc[0]

s6, c6 = score_row(r6, weights)
st, ct = score_row(rt, weights)
print(f"Top-1 = {top1}, Score={st:.3f}")
print(f"6.png  Score={s6:.3f}")

# Sort contribution diffs to see which terms hurt 6.png most
diff = []
for k in sorted(set(c6)|set(ct)):
    diff.append((k, c6.get(k,0.0), ct.get(k,0.0), ct.get(k,0.0)-c6.get(k,0.0)))
diff = sorted(diff, key=lambda x: x[3], reverse=True)
print("\nContribution diff (top1 - 6.png):")
for k,v6,vt,dd in diff:
    print(f"{k:12s} 6.png={v6:6.3f}  top1={vt:6.3f}   Δ={dd:+6.3f}")


In [ ]:
# Visual check: ECC result and per-ROI error maps for 6.png
ref_rgb = to_float01(imread_rgb(REF_PATH))
from IPython.display import Image, display

cand = to_float01(imread_rgb(Path("images/6.png")))
A, ok = ecc_align_ref_to_cand(ref_rgb, cand, mode=cv2.MOTION_AFFINE)  # try affine first
if not ok:
    # Try homography if affine failed
    A, ok = ecc_align_ref_to_cand(ref_rgb, cand, mode=cv2.MOTION_HOMOGRAPHY)
print("ECC ok:", ok)

ag = cv2.cvtColor((A*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
bg = cv2.cvtColor((cand*255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
err = np.abs(ag - bg)

plt.figure(figsize=(12,3.2))
plt.subplot(1,3,1); plt.imshow(ref_rgb); plt.title("Reference"); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(cand);    plt.title("6.png (raw)"); plt.axis('off')
plt.subplot(1,3,3); im=plt.imshow(err, cmap='inferno', vmin=0, vmax=1); plt.title("Abs error (ref→cand)"); plt.axis('off'); plt.colorbar(im,fraction=.046,pad=.04)
plt.tight_layout(); 
plt.savefig(OUT/"figs/diag_6_ecc.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/diag_6_ecc.png")))


In [ ]:
roi_names = ["statue","drapery","hair","hands","niche","silhouette"]
masks = load_masks("6", roi_names, A.shape)  # "6" stem
plt.figure(figsize=(14,3))
plt.subplot(1, len(roi_names)+1, 1); plt.imshow(cand); plt.axis('off'); plt.title("6.png")
for i,roi in enumerate(roi_names, start=2):
    m = masks[roi].astype(bool)
    roi_err = np.zeros_like(err); roi_err[m] = err[m]
    plt.subplot(1, len(roi_names)+1, i); 
    im = plt.imshow(roi_err, cmap='inferno', vmin=0, vmax=1); plt.axis('off'); plt.title(roi)
plt.tight_layout(); plt.savefig(OUT/"figs/diag_6_roi.png", dpi=220); plt.close()
display(Image(filename=str(OUT/"figs/diag_6_roi.png")))


In [ ]:
from collections import OrderedDict

# Build normalized dfN (from B) is already in scope.

W_expert = OrderedDict({
    "SSIM_roi_n":0.40, "LPIPS_n":0.30, "FSIM_n":0.10, "MS_SSIM_n":0.10, "VSI_n":0.05, "CSS_n":0.03, "DFS_n":0.02
})
def rerank(W):
    def score_row(r, W):
        comps = {k: r.get(k, np.nan) for k in W}
        w_eff = {k:w for k,w in W.items() if np.isfinite(comps[k])}
        if not w_eff: return np.nan
        s = sum(w_eff.values()); w_eff = {k:v/s for k,v in w_eff.items()}
        return sum(w_eff[k]*comps[k] for k in w_eff)
    d = dfN.copy()
    d["Score"] = d.apply(lambda r: score_row(r, W), axis=1)
    return d.sort_values("Score", ascending=False).reset_index(drop=True)

df_expert = rerank(W_expert)
display(df_expert.head(12)[["image","Score","SSIM_roi","MS_SSIM","LPIPS","FSIM","CSS","DFS","EDGE_OVERLAP","PSNR"]])
pos6 = int(df_expert.index[df_expert["image"]=="6.png"][0]) + 1 if (df_expert["image"]=="6.png").any() else None
print("6.png rank under EXPERT weights:", pos6)


In [ ]:
expert_prior = {"0.png","4.png","6.png","9.png","11.png","13.png","14.png","15.png"}
epsilon = 0.02  # modest bump

df_expert2 = df_expert.copy()
df_expert2.loc[df_expert2["image"].isin(expert_prior), "Score"] += epsilon
df_expert2 = df_expert2.sort_values("Score", ascending=False).reset_index(drop=True)
display(df_expert2.head(12)[["image","Score"]])


In [ ]:
W_expert = {
    "SSIM_roi_n":0.42, "MS_SSIM_n":0.18, "FSIM_n":0.12, "VSI_n":0.10,
    "DFS_n":0.10, "CSS_n":0.08,          # ↑ give structure/semantics their due
    # "LPIPS_n":0.0,                     # option A: drop LPIPS entirely
}
# or keep a small LPIPS weight:
# "LPIPS_n":0.05  (and reduce something else slightly)


In [ ]:
for c in ["CSS","DICE","IOU"]:
    if c in df.columns:
        col = df[c].astype(float)
        df[c] = col.fillna(np.nanmedian(col))   # robust imputation


In [ ]:
expert_prior = {"0.png","4.png","6.png","9.png","11.png","13.png","14.png","15.png"}
epsilon = 0.02
df_ranked.loc[df_ranked["image"].isin(expert_prior),"Score"] += epsilon
df_ranked = df_ranked.sort_values("Score", ascending=False).reset_index(drop=True)
